# Modules mapping
Map modules into a graph that has the version of the mhb it is part of as weights (for the nodes)

In [1]:
import os
import sys
os.environ['PYTHONPATH'] = os.path.expanduser('~/mhbai')
sys.path.append(os.path.expanduser('~/mhbai'))

In [23]:
from psycopg import sql
from database import database as db


query = sql.SQL("""
    WITH a AS (
    SELECT
        ai.raw_module_id AS ai_raw_module_id,
        ai.title AS ai_title,
        ai.module_code AS ai_module_code,
        ai.ects AS ai_ects,
        ai.lecturer AS ai_lecturer,
        ai.contents AS ai_contents,
        ai.goals AS ai_goals,
        ai.requirements AS ai_requirements,
        ai.expense AS ai_expense,
        ai.success_requirements AS ai_success_requirements,
        ai.weekly_hours AS ai_weekly_hours,
        ai.recommended_semester AS ai_recommended_semester,
        ai.exams AS ai_exams,
        ai.module_parts AS ai_module_parts,

        modules.id AS module_id, 
        modules.title AS module_title, 
        modules.module_code AS module_code, 
        modules.ects AS module_ects, 
        modules.content AS module_content, 
        modules.goals AS module_goals, 
        modules.pages AS module_pages, 
        modules.mhb_id AS module_mhb_id, 

        raw.content AS raw_content,
                
        mhbs.version AS version,

        CASE 
            WHEN ai.module_code IS NULL THEN 0.5
            WHEN ai.module_code = modules.module_code THEN 1
            ELSE 0
        END AS correct_module_code,
        CASE 
            WHEN ai.ects IS NULL THEN 0.5
            WHEN ai.ects = modules.ects THEN 1
            ELSE 0
        END AS correct_ects,
        CASE 
            WHEN ai.title IS NULL THEN 0.5
            WHEN ai.title = modules.title THEN 1
            ELSE 0
        END AS correct_title,
        ai.id AS ai_id,
        mhbs.id AS mhb_id,
        mhbs.pdf_name AS pdf_name,
        modules.id AS module_id

    FROM unia.modules_ai_extracted AS ai
    JOIN unia.modules_raw AS raw ON ai.raw_module_id = raw.id
    JOIN unia.modules ON raw.mhb_id = modules.mhb_id AND raw.module_code = modules.module_code
    JOIN unia.mhbs ON modules.mhb_id = mhbs.id
    )
    SELECT * FROM a ORDER BY correct_module_code + correct_ects + correct_title DESC
    ;
    """).format()

# execute query
result = db.custom_call(
    query=query,
    type_of_answer=db.ANSWER_TYPE.LIST_ANSWER,
)

# check for errors
if result.is_error:
    raise UserWarning("Error fetching data:", result.error)

# initialize modules list and define columns for mapping query results
modules = []
columns = [
    "raw_module_id",
    "ai_title",
    "ai_module_code",
    "ai_ects",
    "ai_lecturer",
    "ai_contents",
    "ai_goals",
    "ai_requirements",
    "ai_expense",
    "ai_success_requirements",
    "ai_weekly_hours",
    "ai_recommended_semester",
    "ai_exams",
    "ai_module_parts",

    "module_id",
    "module_title",
    "module_code",
    "module_ects",
    "module_content",
    "module_goals",
    "module_pages",
    "module_mhb_id",

    "raw_content",

    "version",

    "correct_module_code",
    "correct_ects",
    "correct_title",
    "ai_id",
    "mhb_id",
    "pdf_name",
    "module_id"
]

# map query results to module information
for mod in result.data:
    raw_module = {key: value for key, value in zip(columns, mod)}

    # could be handled in sql (maybe is)
    if raw_module["module_code"] is None and raw_module["ai_module_code"] is None:
        continue

    # build module dict with ai and raw data, using ai data if raw data is None, and calculate correctness scores
    module = {
        "module_code": raw_module["ai_module_code"] if raw_module["module_code"] is None else raw_module["module_code"],
        "title": raw_module["ai_title"] if raw_module["module_title"] is None else raw_module["module_title"],
        "ects": raw_module["ai_ects"] if raw_module["module_ects"] is None else raw_module["module_ects"],
        "lecturer": raw_module["ai_lecturer"],
        "contents": raw_module["ai_contents"],
        "goals": raw_module["ai_goals"],
        "requirements": raw_module["ai_requirements"],
        "expense": raw_module["ai_expense"],
        "success_requirements": raw_module["ai_success_requirements"],
        "weekly_hours": raw_module["ai_weekly_hours"],
        "recommended_semester": raw_module["ai_recommended_semester"],
        "exams": raw_module["ai_exams"],
        "module_parts": raw_module["ai_module_parts"],
        "version": ("Winter" if (winter := int((version := str(raw_module["version"]))[-1]) == 1) else "Sommer") + "semester " + (year := version[:-1]) + (("/" + str(int(year) + 1)) if winter else ""),
        "correct_module_code": float(raw_module["correct_module_code"]),
        "correct_ects": float(raw_module["correct_ects"]),
        "correct_title": float(raw_module["correct_title"]),
        "ai_id": raw_module["ai_id"],
        "mhb_id": raw_module["mhb_id"],
        "pdf_name": raw_module["pdf_name"],
        "module_id": raw_module["module_id"],
        "confidence_score": float((raw_module["correct_module_code"] + raw_module["correct_ects"] + raw_module["correct_title"]) / 3)
    }

    modules.append(module)

In [24]:
def is_valid(module: dict) -> bool:
    """
    Checks if a module is valid based on its confidence score and the presence of required fields.

    Args:
        module (dict): The module to check.
    
    Returns:
        bool: True if the module is valid, False otherwise.
    """
    return (
        module["confidence_score"] == 1 and
        module["contents"] is not None and module["contents"] != [] and
        module["goals"] is not None and module["goals"] != [] and
        module["module_code"] is not None
    )

In [46]:
modules = [item for item in modules if is_valid(item)]
for i in modules:
    i["course_name"] = (name := i["pdf_name"].split("__", 1)[1])[:min(i for i in [name.find("_PO"), name.find("_ID")] if i != -1)]
    if i["course_name"].startswith("PO_"):
        i["course_name"] = i["course_name"][8:]
modules = [i for i in modules if len(i["course_name"]) > 8]

In [72]:
print(len(modules))
mods = [dict((k, v) for k, v in i.items() if k not in ["ai_id", "mhb_id", "module_id"]) for i in modules]
print(len(set(i["module_code"] + i["course_name"] for i in modules)))
print(len(set(i["module_code"] for i in modules)))
print(len(set(i["module_code"] + i["course_name"] for i in modules)) / len(set(i["module_code"] for i in modules)))

13808
6623
2620
2.5278625954198475


# Graph structure (explicit)
    Study Program
    Module Handbook
    Module

In [63]:
for i in set(i["course_name"] for i in modules):
    print(i)

MSc_Informatik_und_Multimedia
Lehramt_an_Mittelschulen_LPO_UA_2023_Unterrichtsfach_Katholische_Religionslehre
Lehramtsbezogener_Bachelorstudiengang_Gymnasium
Zusatzqualifikation_Klassenmusizieren
Studiengang_Lehramt_HauptMittelschule_Mathematik_LPO_2012_Version_ab_WS_2015
Lehramt_an_Grundschulen_LPO_UA_2023_Unterrichtsfach_Physik
Evangelische_Religionslehre_Lehramt_Realschule_LPO_UA_2012
BA_FrankoRomanistik_HF
Masterstudiengang_Anwendungsorientierte_Interkulturelle_Sprachwissenschaft_ANIS
Lehramt_Gymnasium_Geographie_LPO2008
BA_Bachelor_of_Arts_Nebenfach_Musikwissenschaft
Studiengang_Lehramt_Realschule_LPO_2012_Version_ab_WS_2015
BSc_Ingenieurinformatik
NordamerikaStudien
Masterstudiengang_Interdisziplinaere_Europastudien_Studienbeginn_ab_WS_1718
Bachelor_of_Arts_AnglistikAmerikanistik_Hauptfach_BaPO_2008
Lehramt_an_Realschulen_LPO_UA_2023_Unterrichtsfach_Physik
Bachelor_Medien_und_Kommunikation
Master_of_Arts_Kunstpaedagogik
Masterstudiengang_Mathematik
Freier_Bereich_im_Lehramt_an_Mi

In [50]:
print(f"Percentage of unique courses: {len(set(i['course_name'] for i in modules)) / len(modules) * 100:.2f}%")
print(len(set(i["course_name"] for i in modules)), len(modules))

Percentage of unique courses: 2.10%
290 13808


In [ ]:
distinct_courses = 

-1